# 融合算子优化与 Profiling

第 3 章我们用基线 SFT 配置完成了 Wordle 模型训练，并保存了训练日志与 checkpoint。

但训练性能还有优化空间。本章从性能瓶颈分析出发，引入 Ascend NPU 融合算子，通过 ModelConverter 一行配置接入，最后用 profiling 数据验证优化效果。

---

## 教程进度回顾

| 章节 | 内容 | 状态 |
|------|------|------|
| 第 1 章 | SFT 概念 + Wordle 任务 | ✅ 已完成 |
| 第 2 章 | TorchTitan 框架 + 环境配置 | ✅ 已完成 |
| 第 3 章 | 数据准备 + 基线训练 + 推理评测 | ✅ 已完成 |
| **第 4 章** | **融合算子 + Profiling 优化** | ← 当前 |

---

## 基线性能回顾：瓶颈在哪？

本章使用与第 3 章相同的模型、batch、序列长度和两个 rank，分别运行 baseline 与 fused 配置。训练日志提供 step time、tps、MFU 和显存；profile trace 再区分算子计算、通信/同步与数据等待。低 MFU 可能来自其中任一类瓶颈，需要结合日志和 trace 定位。

### 算子碎片化

以 **RMSNorm** 为例：当前模型中的匹配数量由 converter 使用的类型和 `model.named_modules()` 输出决定。原生路径会产生多少 device kernel 也取决于 PyTorch、torch_npu、shape、dtype 和编译配置，因此本章从本次 trace 读取实际调用名称、数量和耗时。

---

## 解决方案

用 Ascend NPU 融合算子 `torch_npu.npu_rms_norm` 替换原生 RMSNorm 路径，具体实现在 04.02 详述。本章随后比较 baseline 与 fused 的 trace 和端到端训练指标。

---

## 前置条件

- 第 2 章环境已配置，`torch_npu` 可正常 `import`。
- 第 3 章基线训练命令可以正常完成。

---

## 本章结构

| Notebook | 内容 |
|---|---|
| [04.01](04.01_chapter_intro.ipynb) | 章节介绍（本节） |
| [04.02](04.02_adding_fused_operator.ipynb) | 融合算子实现：`rms_norm.py` 代码详解 |
| [04.03](04.03_running_and_profiling.ipynb) | Profiling 实操：跑 baseline vs fused |
| [04.04](04.04_profiling_output_analysis.ipynb) | Profiling 结果分析 |
| [04.05](04.05_chapter_practice.ipynb) | 章节练习 |

下一节，我们将阅读 `torchtitan_npu/converters/kernels/rms_norm.py` 的关键实现。

## 练习

1. （单选题）当前模型的 RMSNorm 匹配数量应从哪里获得？
    A. converter 使用的匹配类型、`named_modules()` 名称清单和总数
    B. 固定使用 `28 × 2`
    C. checkpoint 文件大小
    D. tokenizer 词表大小

2. （单选题）哪组结果分别回答“融合路径生效”和“端到端训练变快”？
    A. fused trace 出现 `npu_rms_norm`；未启用 profiler 的重复 step time/tps 对比
    B. loss 下降；模型参数量不变
    C. tokenizer 可加载；checkpoint 存在
    D. seq_len 相同；学习率相同

3. （多选题）MFU 偏低时，profile 应区分哪些可能来源？
    A. 算子计算效率
    B. 通信与同步等待
    C. 数据加载或 host gap
    D. 只检查学习率

In [ ]:
!cat cann-learning-hub/tutorials/sft_training_pipeline/04_fused_operators/answer/04.01_answer.txt